## OpenAI Luna

In [1]:
!pip install langchain-openai==1.7.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 7.2 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1")

llm.invoke("hi").content

'Hello! How can I help you today? 😊'

In [4]:
# import os
# from openai import OpenAI

# # It will automatically pick up the OPENAI_API_KEY from your environment variables
# client = OpenAI(
#     api_key=os.environ.get("OPENAI_API_KEY")
# )

# # Fetch all models accessible to this specific API key context
# models = client.models.list()

# # Print the model IDs alphabetically
# model_list = sorted([model.id for model in models])
# for model_id in model_list:
#     print(model_id)


In [6]:
from langchain_openai.decisions import (
    Choice,
    Level,
    OpenAIDecisions,
    Predicate,
    Score,
)

decisions = OpenAIDecisions(model="gpt-6-luna")

response = decisions.invoke(
    {
        "input": (
            "The deploy failed twice and customers are seeing 500s. "
            "Can someone look now?"
        ),
        "questions": {
            "urgent": Predicate(instructions="Does this need attention right now?"),
            "team": Choice(
                instructions="Which team should pick this up?",
                choices={
                    "infra": "Deploys, availability, and on-call incidents.",
                    "billing": "Payments, invoices, and subscriptions.",
                },
            ),
            "severity": Score(
                instructions="How severe is the impact?",
                levels=[
                    "Cosmetic",
                    Level(label="Degraded", description="Degraded for some users."),
                    Level(label="Outage", description="Full outage."),
                ],
            ),
        },
    }
)

In [8]:
print(response.predicates["urgent"].probability)
print(response.choices["team"].choice, response.choices["team"].confidence)
print(response.scores["severity"].score)

1.0
infra 1.0
1.0


In [10]:
response.model_dump()

{'model': 'gpt-6-luna',
 'answers': {'urgent': {'type': 'predicate', 'probability': 1.0},
  'team': {'type': 'choice',
   'choice': 'infra',
   'probabilities': {'infra': 1.0, 'billing': 0.0},
   'confidence': 1.0},
  'severity': {'type': 'score',
   'score': 1.0,
   'legend': {0: 'Cosmetic', 1: 'Degraded', 2: 'Outage'},
   'probabilities': {0: 0.01, 1: 0.98, 2: 0.01},
   'confidence': 0.97}},
 'usage': {'input_tokens': 411, 'output_tokens': 0},
 'request_id': 'req_533474eb1435487abe9d4ac6442da75f'}